In [1]:
import sys
import json
import html
import sqlite3
import importlib

from pathlib import Path
from contextlib import closing
from IPython.display import display, HTML

ROOT = Path(r"C:\ai_campus_main_proj\rss_news")
SRC = ROOT / "src"

if not (SRC / "rss_pipeline.py").is_file():
    raise FileNotFoundError("프로젝트 경로를 확인하세요.")

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import npr_ieee_rules
import integrated_notebook_bodies as rules
import rss_pipeline as pipeline

for module in (npr_ieee_rules, rules, pipeline):
    importlib.reload(module)

print("준비 완료:", ROOT)

준비 완료: C:\ai_campus_main_proj\rss_news


In [2]:
FEEDS = pipeline.load_feeds()
SOURCES = {feed["source_name"] for feed in FEEDS}

BODY_LIMIT = 1  # 실행당 매체별 본문 처리 상한
DB_PATH = ROOT / "data" / "combined_rss.sqlite3"

print(f"수집 대상: {len(SOURCES)}개 매체·{len(FEEDS)}개 피드")
print("WIRED 제외:", "WIRED" not in SOURCES)

rules.show_report(rules.source_rules())

수집 대상: 19개 매체·36개 피드
WIRED 제외: True


매체,상태,비고
CNBC,본문 추출 표본 확인,
Fortune,통합 실행에서 본문 후보 확보·전문 미검증,
The Motley Fool,통합 실행에서 본문 후보 확보·전문 미검증,
Zacks,본문 보류,과거 차단·본문 누락의 범용 해결 미완료: RSS만 유지
Fox Business,확인한 추천·캡션·홍보 정제 반영,
Business Insider,본문 위치 수정·구독으로 대조 불가한 표본 있음,
BBC News,본문 추출 표본 확인,
The Guardian,본문 추출 표본 확인,
Ars Technica,본문 위치 수정·사용자 표본 확인,
TechCrunch,본문 추출 표본 확인,


In [3]:
result = pipeline.collect_once(
    collect=True,
    db_path=DB_PATH,
    body_per_source=BODY_LIMIT,
    delay=2,
)

print("\n수집 완료")
print("새 항목:", result["new_articles"])
print("RSS 실패:", result["feed_failures"])
print("본문 요청:", result["body_attempts"])
print("본문 상태:", result["body_states"])

rules.show_report(result["media"])

RSS 1/36: CNBC / top
RSS 2/36: CNBC / business
RSS 3/36: CNBC / economy
RSS 4/36: CNBC / finance
RSS 5/36: CNBC / technology
RSS 6/36: CNBC / world
RSS 7/36: Fortune / all
RSS 8/36: The Motley Fool / headlines
RSS 9/36: Zacks / commentary
RSS 10/36: Fox Business / latest
RSS 11/36: Fox Business / economy
RSS 12/36: Fox Business / markets
RSS 13/36: Fox Business / technology
RSS 14/36: Business Insider / legacy_atom
RSS 15/36: BBC News / top
RSS 16/36: BBC News / business
RSS 17/36: BBC News / technology
RSS 18/36: The Guardian / world
RSS 19/36: The Guardian / business
RSS 20/36: The Guardian / technology
RSS 21/36: Ars Technica / all_news
RSS 22/36: TechCrunch / all
RSS 23/36: NPR / news
RSS 24/36: NPR / business
RSS 25/36: ProPublica / main
RSS 26/36: IEEE Spectrum / all
RSS 27/36: MIT Technology Review / all
RSS 28/36: ABC News / top
RSS 29/36: ABC News / business
RSS 30/36: ABC News / technology
RSS 31/36: CBS News / top
RSS 32/36: CBS News / moneywatch
RSS 33/36: CBS News / techno

매체,누적 항목,본문 후보,대기,비고
CNBC,151,4,142,
Fortune,12,4,8,
The Motley Fool,53,4,49,
Zacks,325,0,0,과거 차단·본문 누락의 범용 해결 미완료: RSS만 유지
Fox Business,94,4,90,
Business Insider,20,4,16,
BBC News,95,4,68,
The Guardian,120,4,104,
Ars Technica,20,4,16,
TechCrunch,20,4,15,


In [4]:
run_dir = Path(result["output"])

records = [
    json.loads(path.read_text(encoding="utf-8"))
    for path in sorted(run_dir.glob("body_*.json"))
]

rows = [{
    "매체": record["source_name"],
    "상태": record["body_status"],
    "본문 글자 수": len(record.get("body_clean", "")),
    "본문 유형": record.get("body_kind", "일반 본문 후보"),
    "IEEE 분류": record.get(
        "content_classification", {}
    ).get("status_label", ""),
    "오류": record.get("error", ""),
} for record in records]

rules.show_report(rows)

print("상세 기록:", run_dir)

매체,상태,본문 글자 수,본문 유형,IEEE 분류,오류
CNBC,candidate_review,3392,일반 본문 후보,,
Fortune,integration_review,6310,일반 본문 후보,,
The Motley Fool,integration_review,4595,일반 본문 후보,,
Fox Business,integration_review,6063,일반 본문 후보,,
Business Insider,integration_review,5397,일반 본문 후보,,
BBC News,candidate_review,6207,일반 본문 후보,,
The Guardian,candidate_review,4846,일반 본문 후보,,
Ars Technica,integration_review,4271,일반 본문 후보,,
TechCrunch,candidate_review,1152,일반 본문 후보,,
NPR,integration_review,4779,일반 본문 후보,,


상세 기록: C:\ai_campus_main_proj\rss_news\outputs\combined_rss\20261105T074038_434409Z


In [5]:
if not DB_PATH.is_file():
    print("저장된 DB가 없습니다.")

else:
    placeholders = ",".join("?" for _ in SOURCES)

    with closing(sqlite3.connect(
        DB_PATH.resolve().as_uri() + "?mode=ro",
        uri=True,
    )) as conn:
        conn.row_factory = sqlite3.Row

        articles = conn.execute(f"""
            SELECT source_name, title, url, published,
                   body_status, body_clean
            FROM articles
            WHERE source_name IN ({placeholders})
              AND LENGTH(TRIM(COALESCE(body_clean, ''))) > 0
            ORDER BY body_checked_at_utc DESC
            LIMIT 50
        """, sorted(SOURCES)).fetchall()

    cards = []

    for article in articles:
        source = html.escape(article["source_name"])
        title = html.escape(article["title"] or "제목 없음")
        url = html.escape(article["url"], quote=True)
        body = html.escape(article["body_clean"])
        status = html.escape(article["body_status"])
        published = html.escape(article["published"] or "미확인")

        cards.append(
            '<details style="border:1px solid #888;'
            'padding:14px;margin:10px 0;">'
            '<summary style="cursor:pointer;font-weight:bold;">'
            f'{source} — {title}</summary>'
            f'<p><a href="{url}" target="_blank" '
            'rel="noopener noreferrer">원문 열기</a></p>'
            f'<p>발행일: {published}</p>'
            f'<p>상태: {status} / 전문 여부 미검증</p>'
            '<div style="white-space:pre-wrap;max-height:550px;'
            'overflow:auto;border-top:1px solid #888;padding-top:12px;">'
            f'{body}</div></details>'
        )

    display(HTML(
        "".join(cards) or "본문이 저장된 기사가 없습니다."
    ))

# RSS 통합 수집·본문 정제

- 최초 조사: 20개 매체·38개 피드
- 현재 수집: WIRED 제외, 19개 매체·36개 피드
- Zacks·MIT: RSS만 수집, 본문 자동 요청 보류
- 매체별 본문 추출·정제, NPR 공개 대본 탐색, IEEE 시문예 분류 적용
- ProPublica 제보 안내는 유지
- 본문 후보 확보는 전문·정제 정확성 검증 완료를 뜻하지 않음
- 실행당 본문 처리 상한을 넘은 기사는 대기 상태로 누적
- ③번 셀은 실제 수집, ④·⑤번 셀은 저장 결과 조회